In [1]:
import pandas as pd

sales = pd.read_csv("../data/sample/retail_sales.csv")

sales.head()

,order_id,order_date,country,category,units_sold,unit_price_eur
0,1001,2026-06-01,Germany,Electronics,2,99.99
1,1002,2026-06-01,Germany,Home,4,24.50
2,1003,2026-06-02,France,Electronics,1,249.99
3,1004,2026-06-02,France,Home,6,18.75
4,1005,2026-06-03,Netherlands,Electronics,3,79.99


In [4]:
sales.shape # get number of rows and columns

(10, 6)

In [ ]:
sales.info() # column names, data types, non null values, memory information

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        10 non-null     int64  
 1   order_date      10 non-null     str    
 2   country         10 non-null     str    
 3   category        10 non-null     str    
 4   units_sold      10 non-null     int64  
 5   unit_price_eur  10 non-null     float64
dtypes: float64(1), int64(2), str(3)
memory usage: 612.0 bytes


In [ ]:
sales.describe() # summarize numeric columns

,order_id,units_sold,unit_price_eur
count,10.00000,10.00000,10.00000
mean,1005.50000,2.90000,120.07000
std,3.02765,1.66333,151.26056
min,1001.00000,1.00000,18.75000
25%,1003.25000,2.00000,27.00000
50%,1005.50000,2.50000,60.49500
75%,1007.75000,3.75000,122.49000
max,1010.00000,6.00000,499.99000


In [ ]:
sales.isna().sum() # count missing values per column

order_id          0
order_date        0
country           0
category          0
units_sold        0
unit_price_eur    0
dtype: int64

In [ ]:
sales[['country', 'category', 'units_sold']] # select columns

,country,category,units_sold
0,Germany,Electronics,2
1,Germany,Home,4
2,France,Electronics,1
3,France,Home,6
4,Netherlands,Electronics,3
5,Germany,Home,2
6,Netherlands,Home,5
7,France,Electronics,2
8,Germany,Electronics,1
9,Netherlands,Home,3


In [ ]:
sales[sales['country'] == 'Germany'] # filter rows

,order_id,order_date,country,category,units_sold,unit_price_eur
0,1001,2026-06-01,Germany,Electronics,2,99.99
1,1002,2026-06-01,Germany,Home,4,24.50
5,1006,2026-06-03,Germany,Home,2,34.50
8,1009,2026-06-05,Germany,Electronics,1,499.99


In [10]:
electronicsSales = sales[sales['category'] == 'Electronics']
electronicsSales

,order_id,order_date,country,category,units_sold,unit_price_eur
0,1001,2026-06-01,Germany,Electronics,2,99.99
2,1003,2026-06-02,France,Electronics,1,249.99
4,1005,2026-06-03,Netherlands,Electronics,3,79.99
7,1008,2026-06-04,France,Electronics,2,129.99
8,1009,2026-06-05,Germany,Electronics,1,499.99


In [12]:
sales['revenue_eur'] = sales['units_sold'] * sales['unit_price_eur']
sales.head()

,order_id,order_date,country,category,units_sold,unit_price_eur,revenue_eur
0,1001,2026-06-01,Germany,Electronics,2,99.99,199.98
1,1002,2026-06-01,Germany,Home,4,24.50,98.00
2,1003,2026-06-02,France,Electronics,1,249.99,249.99
3,1004,2026-06-02,France,Home,6,18.75,112.50
4,1005,2026-06-03,Netherlands,Electronics,3,79.99,239.97


In [ ]:
country_summary = (
    sales
    .groupby("country", as_index=False) # as index False means country will stay as a normal visible column, else country will be special row label index
    .agg(
        total_revenue_eur=("revenue_eur", "sum"), # describe columns you want in aggregation --> sum, min, max, nunique..
        total_units_sold=("units_sold", "sum"),
        number_of_orders=("order_id", "nunique")
    )
    .sort_values("total_revenue_eur", ascending=False)
)
# groupby --> aggregate --> sort 
country_summary

,country,total_revenue_eur,total_units_sold,number_of_orders
1,Germany,866.97,9,4
0,France,622.47,9,3
2,Netherlands,472.97,11,3


In [14]:
country_summary.to_csv("../data/processed/country_revenue_summary.csv", index= False) # index false means no additional row for index

In [ ]:
import duckdb # use SQL for the same operation

In [17]:
duckdb.sql("""
           SELECT
           country,
           SUM(units_sold * unit_price_eur) AS total_revenue_eur,
           SUM(units_sold) AS total_units_sold,
           COUNT(*) AS number_of_orders
           FROM '../data/sample/retail_sales.csv'
           GROUP BY country
           ORDER BY total_revenue_eur DESC
           """).df()

,country,total_revenue_eur,total_units_sold,number_of_orders
0,Germany,866.97,9.0,4
1,France,622.47,9.0,3
2,Netherlands,472.97,11.0,3
